In [ ]:
import pandas as pd
import numpy as np

# =========================
# 0. 파일 경로
# =========================
PRICE_FILE = "3년수정주가.xlsx"
ROE_FILE = "3년분기ROE.xlsx"
DEBT_FILE = "3년분기부채비율.xlsx"
OP_FILE = "3년분기영업이익.xlsx"

# =========================
# 1. 전략 파라미터
# =========================
N_HOLDINGS = 15

ROE_THRESHOLD = 20
DEBT_THRESHOLD = 200

MOM_SHORT_WINDOW = 10
MOM_LONG_WINDOW = 60

FUND_LAG_DAYS = 0

# =========================
# 2. 데이터 로더
# =========================
def load_price_data(path):
    df = pd.read_excel(path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.set_index("Date").sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def load_quarter_data_simple(path):
    df = pd.read_excel(path)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.set_index("Date").sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

def expand_quarter_to_daily(q_df, daily_index, lag_days=0):
    daily = q_df.reindex(daily_index).ffill()
    if lag_days > 0:
        daily = daily.shift(lag_days)
    return daily

# =========================
# 3. 데이터 불러오기
# =========================
price = load_price_data(PRICE_FILE)
roe_q = load_quarter_data_simple(ROE_FILE)
debt_q = load_quarter_data_simple(DEBT_FILE)
op_q = load_quarter_data_simple(OP_FILE)

# 공통 종목만 사용
common_cols = sorted(list(set(price.columns) & set(roe_q.columns) & set(debt_q.columns) & set(op_q.columns)))
price = price[common_cols]
roe_q = roe_q[common_cols]
debt_q = debt_q[common_cols]
op_q = op_q[common_cols]

# =========================
# 4. 일별 데이터 확장
# =========================
roe_daily = expand_quarter_to_daily(roe_q, price.index, FUND_LAG_DAYS)
debt_daily = expand_quarter_to_daily(debt_q, price.index, FUND_LAG_DAYS)

op_positive_q = op_q > 0
op_2q_positive_q = op_positive_q & op_positive_q.shift(1)
op_2q_positive_daily = expand_quarter_to_daily(
    op_2q_positive_q.astype(float),
    price.index,
    FUND_LAG_DAYS
) > 0

# =========================
# 5. 최신 기준일
# =========================
latest_date = price.index[-1]

# 충분한 모멘텀 기간이 있는지 확인
if len(price) < max(MOM_SHORT_WINDOW, MOM_LONG_WINDOW) + 1:
    raise ValueError("가격 데이터 길이가 모멘텀 계산에 부족합니다.")

# =========================
# 6. 모멘텀 계산
# =========================
mom_short = price / price.shift(MOM_SHORT_WINDOW) - 1
mom_long = price / price.shift(MOM_LONG_WINDOW) - 1

# =========================
# 7. 현재 시점 스크리닝
# =========================
q = (
    (roe_daily.loc[latest_date] > ROE_THRESHOLD) &
    (debt_daily.loc[latest_date] < DEBT_THRESHOLD) &
    (op_2q_positive_daily.loc[latest_date])
)

ms = mom_short.loc[latest_date]
ml = mom_long.loc[latest_date]

final_filter = q & (ms > 0) & (ml > 0)
candidates = ms[final_filter].dropna()

if len(candidates) == 0:
    print("조건을 만족하는 종목이 없습니다.")
else:
    top = candidates.sort_values(ascending=False).head(N_HOLDINGS)
    selected = top.index.tolist()

    result_df = pd.DataFrame({
        "종목": selected,
        "10일 모멘텀(%)": (ms[selected] * 100).values,
        "60일 모멘텀(%)": (ml[selected] * 100).values,
        "ROE(%)": roe_daily.loc[latest_date, selected].values,
        "부채비율(%)": debt_daily.loc[latest_date, selected].values,
        "최근2분기영업이익": op_2q_positive_daily.loc[latest_date, selected].values,
        "비중(%)": round(100 / len(selected), 2)
    })

    result_df = result_df.sort_values("10일 모멘텀(%)", ascending=False).reset_index(drop=True)

    for col in ["10일 모멘텀(%)", "60일 모멘텀(%)", "ROE(%)", "부채비율(%)"]:
        result_df[col] = result_df[col].round(2)

print(f"\n기준일: {latest_date.date()}")
print("\n현재 포트폴리오 스크리닝 결과")
result_df